<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Boosting Intuition</b>
</h1>
<div style="font-family:'Times New Roman';">
Random forest was bagging, lots of trees trained in parallel and then they vote. Boosting is the other big ensemble idea, but it works in a totally diffrent way. Instead of training all the models independently, boosting trains them one after another, and each new model tries to fix the mistakes the previous ones made. Somehow stacking up a bunch of weak models like this gives you one really strong model.
</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_gaussian_quantiles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## Bagging vs Boosting

Both build many models and combine them, but the how is different:

- **Bagging (random forest)** : train lots of deep trees in parallel on random subsets, then average their votes. It mainly reduces variance.
- **Boosting** : train small weak models one at a time, in sequence, where each one focuses on the points the earlier ones got wrong. It mainly reduces bias.

So bagging is many strong learners voting, boosting is many weak learners teaming up in a chain.

In [2]:
# a dataset where the two classes sit in rings, so one simple split cant separate them
X, y = make_gaussian_quantiles(n_samples=500, n_features=2, n_classes=2, random_state=42)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
plt.title('two classes in rings')
plt.show()

## What is a weak learner

A weak learner is a model thats only slightly better than guessing. The usual choice for boosting is a **decision stump**, which is just a decision tree with depth 1, so it makes a single split on one feature. On its own a stump is pretty useless for this ring data, lets see how bad.

In [4]:
def plot_regions(predict_fn, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = predict_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
    plt.title(title)
    plt.show()

stump = DecisionTreeClassifier(max_depth=1).fit(X, y)
print('single stump accuracy:', accuracy_score(y, stump.predict(X)).round(3))
plot_regions(stump.predict, X, y, 'one decision stump (weak)')

## The boosting idea

Yeah, one stump is around 60 something percent, basically a coin flip with extra steps. Here is the clever part of boosting:

1. train a stump
2. look at which points it got wrong and give those points more weight
3. train the next stump, which now cares more about those hard points
4. keep going, and at the end let all the stumps vote, but give a bigger say to the stumps that were more accurate

Lets just compare 1 stump against 50 boosted stumps and see the difference.

In [5]:
for n in [1, 50]:
    model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                               n_estimators=n, algorithm='SAMME', random_state=42)
    model.fit(X, y)
    print(f'{n:>2} stumps  accuracy = {accuracy_score(y, model.predict(X)).round(3)}')

plot_regions(model.predict, X, y, '50 boosted stumps (strong)')

So a single stump barely beats guessing, but 50 of them boosted together wrap right around the ring and get the data almost perfect. Thats boosting in a nutshell.

Next i'll actually build AdaBoost myself so the weight updating isnt just a magic sklearn call.